# Equivalence tests, grid sensitivity and the per-fuzzifier design

**Computes:** Hodges-Lehmann estimates, equivalence tests, restricted grids, the share of settings at the grid's edge and the per-fuzzifier comparison design.

**Inputs:** checkpoints of 06 and 07.  
**Outputs:** `results/09/results_09.json` and a text report.


In [ ]:
import os, sys, time, json, pickle, platform, subprocess, glob, urllib.request
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATASETS = None
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")

RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.join(RESULTS_DIR, "09")
OUTDIR = os.path.abspath(OUTDIR)
for sub in ("data", "reports"):
    os.makedirs(os.path.join(OUTDIR, sub), exist_ok=True)
import numpy as np, scipy, sklearn
print(f"colab {IN_COLAB} | python {platform.python_version()} | numpy {np.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}")
print(f"output dir: {OUTDIR}   run stamp: {RUN_STAMP}")
BASE = os.path.dirname(OUTDIR)
DIR_06 = os.environ.get("DIR_06", os.path.join(BASE, "06"))
DIR_07 = os.environ.get("DIR_07", os.path.join(BASE, "07"))
n06 = len(glob.glob(os.path.join(DIR_06, "runs06_*.pkl"))); n07 = len(glob.glob(os.path.join(DIR_07, "runs07_*.pkl")))
print(f"06 checkpoints: {n06} (31 expected) in {DIR_06}\n07 checkpoints: {n07} (31 expected) in {DIR_07}")
assert n06 and n07, "checkpoints of 06/07 not found"

In [ ]:
import numpy as np
from itertools import permutations

EPS = 1e-12

def fcm_membership(D, m):
    p = 2.0 / (m - 1.0)
    z = -p * np.log(np.maximum(D, EPS))
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def hr_bounds(D, m1, m2):
    C = D.shape[1]
    U1, U2 = fcm_membership(D, m1), fcm_membership(D, m2)
    s = 1.0 / (D[:, :, None] / np.maximum(D[:, None, :], EPS)).sum(axis=2)
    cond = s < 1.0 / C
    upper = np.where(cond, U1, U2)
    lower = np.where(cond, U2, U1)
    return lower, upper

def km_interval_1d(x, wl, wu):
    o = np.argsort(x, kind="stable"); xs = x[o]; l = wl[o]; u = wu[o]
    n = len(x)
    cl, cu = np.cumsum(l), np.cumsum(u)
    clx, cux = np.cumsum(l * xs), np.cumsum(u * xs)
    Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
    k = np.arange(n + 1)
    Lk = np.concatenate([[0.0], cl]); Uk = np.concatenate([[0.0], cu])
    Lxk = np.concatenate([[0.0], clx]); Uxk = np.concatenate([[0.0], cux])
    vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
    vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
    kR, kL = int(np.argmax(vR_all)), int(np.argmin(vL_all))
    selR = np.zeros(n, bool); selL = np.zeros(n, bool)
    rank = np.empty(n, int); rank[o] = np.arange(n)
    selR[:] = rank >= kR
    selL[:] = rank < kL
    return float(vL_all[kL]), float(vR_all[kR]), selL, selR

def it2fcm_update(X, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    wl, wu = lower ** mc, upper ** mc
    for j in range(C):
        accL = np.zeros(N); accR = np.zeros(N)
        for l in range(M):
            vL, vR, selL, selR = km_interval_1d(X[:, l], wl[:, j], wu[:, j])
            V[j, l] = 0.5 * (vL + vR)
            accL += np.where(selL, upper[:, j], lower[:, j])
            accR += np.where(selR, upper[:, j], lower[:, j])
        uL[:, j], uR[:, j] = accL / M, accR / M
    return V, uL, uR

def dist(X, V):
    return np.maximum(np.linalg.norm(X[:, None, :] - V[None, :, :], axis=2), EPS)

def run_fcm(X, V0, m, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), m); W = U ** m
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    U = fcm_membership(dist(X, V), m)
    return V, U.argmax(1)

def run_it2fcm(X, V0, m1, m2, mc=None, iters=300, tol=1e-7):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update(X, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update(X, lo, up, mc)
    hard_hr = (0.5 * (uL + uR)).argmax(1)
    hard_nearest = dist(X, V).argmin(1)
    return V, hard_hr, hard_nearest

def accuracy(labels, pred, C):
    best = 0.0
    for perm in permutations(range(C)):
        best = max(best, float(np.mean(np.array(perm)[pred] == labels)))
    return best

def centre_error(V, centres):
    C = len(centres); best = np.inf
    for perm in permutations(range(C)):
        best = min(best, float(np.mean(np.linalg.norm(V[list(perm)] - centres, axis=1))))
    return best

def init_pp(X, C, rng):
    idx = [rng.integers(len(X))]
    for _ in range(1, C):
        d2 = np.min(((X[:, None, :] - X[idx][None]) ** 2).sum(2), axis=1)
        idx.append(rng.choice(len(X), p=d2 / d2.sum()))
    return X[idx].copy()

def scenario(name, seed):
    r = np.random.default_rng(seed)
    if name == "equal":
        c = np.array([[0.0, 0.0], [4.0, 0.0]]); n = (100, 100); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume":
        c = np.array([[0.0, 0.0], [5.0, 0.0]]); n = (100, 100); half = (2.5, 0.6)
        X = np.vstack([c[k] + r.uniform(-half[k], half[k], size=(n[k], 2)) for k in range(2)])
    elif name == "noise":
        c = np.array([[0.0, 0.0], [0.0, 4.0]]); n = (100, 100)
        X = np.vstack([r.normal(c[0], 1.0, size=(n[0], 2)), r.normal(c[1], 1.0, size=(n[1], 2)),
                       r.normal(c[0], 3.0, size=(40, 2))])
        y = np.r_[np.zeros(n[0]), np.ones(n[1]), np.zeros(40)].astype(int)
        return X, y, c, 2
    elif name == "density":
        c = np.array([[0.0, 0.0], [3.5, 0.0]]); n = (200, 30); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume3":
        c = np.array([[0.0, 0.0], [5.0, 0.0], [2.5, 4.0]]); n = (100, 100, 100); s = (1.4, 0.4, 0.8)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(3)])
    else:
        raise ValueError(name)
    y = np.concatenate([np.full(n[k], k) for k in range(len(n))])
    return X, y, c, len(n)

from scipy.optimize import linear_sum_assignment
def accuracy_h(labels, pred, C):
    M = np.zeros((C, C))
    np.add.at(M, (labels, pred), 1)
    r, c = linear_sum_assignment(-M)
    return float(M[r, c].sum() / len(labels))

def align(V, ref):
    Dm = np.linalg.norm(V[:, None] - ref[None], axis=2)
    r, c = linear_sum_assignment(Dm)
    out = np.zeros_like(ref); out[c] = V[r]
    return out

def centre_error_h(V, centres):
    return float(np.mean(np.linalg.norm(align(V, centres) - centres, axis=1)))

def real_dataset(name):
    from sklearn import datasets
    d = {"iris": datasets.load_iris, "wine": datasets.load_wine, "breast_cancer": datasets.load_breast_cancer}[name]()
    X = d.data.astype(float); X = (X - X.mean(0)) / X.std(0)
    y = d.target.astype(int)
    return X, y, len(np.unique(y))

def run_fcm_decoupled(X, V0, ma, mb, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    return V, dist(X, V).argmin(1)

def it2fcm_update_fast(X, order, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    Xs = np.take_along_axis(X, order, axis=0)
    rank = np.empty_like(order); np.put_along_axis(rank, order, np.arange(N)[:, None].repeat(M, 1), axis=0)
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    z = np.zeros((1, M))
    for j in range(C):
        wl = (lower[:, j] ** mc)[order]; wu = (upper[:, j] ** mc)[order]
        cl, cu = np.cumsum(wl, 0), np.cumsum(wu, 0)
        clx, cux = np.cumsum(wl * Xs, 0), np.cumsum(wu * Xs, 0)
        Lk, Uk = np.vstack([z, cl]), np.vstack([z, cu]); Lxk, Uxk = np.vstack([z, clx]), np.vstack([z, cux])
        Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
        vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
        vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
        kR, kL = vR_all.argmax(0), vL_all.argmin(0)
        vR = vR_all[kR, np.arange(M)]; vL = vL_all[kL, np.arange(M)]
        V[j] = 0.5 * (vL + vR)
        selR = rank >= kR[None, :]; selL = rank < kL[None, :]
        up, lo = upper[:, j][:, None], lower[:, j][:, None]
        uR[:, j] = np.where(selR, up, lo).mean(1); uL[:, j] = np.where(selL, up, lo).mean(1)
    return V, uL, uR

def run_it2fcm_fast(X, V0, m1, m2, mc=None, iters=300, tol=1e-7, order=None):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    order = np.argsort(X, axis=0, kind="stable") if order is None else order
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1)

def real_dataset2(name, data_dir="data"):
    import os, pandas as pd
    if name in ("iris", "wine", "wdbc"):
        return real_dataset({"wdbc": "breast_cancer"}.get(name, name))
    if name == "sonar":
        d = pd.read_csv(os.path.join(data_dir, "sonar.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = (d.iloc[:, -1] == "M").to_numpy(int)
    elif name == "pima":
        d = pd.read_csv(os.path.join(data_dir, "pima-indians-diabetes.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = d.iloc[:, -1].to_numpy(int)
    elif name == "wbc":
        d = pd.read_csv(os.path.join(data_dir, "breast-cancer-wisconsin.data"), header=None, na_values="?").dropna()
        X = d.iloc[:, 1:-1].to_numpy(float); y = (d.iloc[:, -1] == 4).to_numpy(int)
    else:
        raise ValueError(name)
    X = (X - X.mean(0)) / X.std(0)
    return X, y, len(np.unique(y))

EXTRA = {"seeds": "wheat-seeds.csv", "glass": "glass.csv", "ionosphere": "ionosphere.csv",
         "banknote": "banknote_authentication.csv", "ecoli": "ecoli.csv"}
def real_dataset3(name, data_dir="data"):
    import os, pandas as pd
    if name not in EXTRA:
        return real_dataset2(name, data_dir)
    d = pd.read_csv(os.path.join(data_dir, EXTRA[name]), header=None)
    X = d.iloc[:, :-1].to_numpy(float)
    y = pd.factorize(d.iloc[:, -1].astype(str).str.strip(), sort=True)[0]
    X = X[:, X.std(0) > 0]
    X = (X - X.mean(0)) / X.std(0)
    return X, y.astype(int), len(np.unique(y))

In [ ]:
import time
import numpy as np

def run_it2fcm_nt(X, V0, m1, m2, mc=None, iters=300, tol=1e-6):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        W = 0.5 * (lo ** mc + up ** mc)
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    return V, (0.5 * (lo + up)).argmax(1)

def contingency(y, p, C):
    M = np.zeros((int(y.max()) + 1, C)); np.add.at(M, (y, p), 1); return M

def ari_from_cont(M):
    comb = lambda x: x * (x - 1) / 2.0
    n = M.sum(); s_ij = comb(M).sum(); a = comb(M.sum(1)).sum(); b = comb(M.sum(0)).sum()
    exp = a * b / comb(n); mx = 0.5 * (a + b)
    return float((s_ij - exp) / (mx - exp)) if mx != exp else 1.0

def nmi_from_cont(M):
    n = M.sum(); P = M / n; pi = P.sum(1); pj = P.sum(0)
    nz = P > 0
    I = float((P[nz] * np.log(P[nz] / (pi[:, None] * pj[None, :])[nz])).sum())
    H = lambda q: float(-(q[q > 0] * np.log(q[q > 0])).sum())
    d = 0.5 * (H(pi) + H(pj))
    return I / d if d > 0 else 1.0

def acc_from_cont(M):
    from scipy.optimize import linear_sum_assignment
    r, c = linear_sum_assignment(-M); return float(M[r, c].sum() / M.sum())

def scores(y, p, C):
    M = contingency(y, p, C)
    return dict(acc=acc_from_cont(M), ari=ari_from_cont(M), nmi=nmi_from_cont(M))

def silhouette_pre(Dsub, lab):
    ks = np.unique(lab)
    if len(ks) < 2: return -1.0
    n = len(lab); a = np.zeros(n); b = np.full(n, np.inf)
    for k in ks:
        mk = lab == k; cnt = mk.sum()
        s = Dsub[:, mk].sum(1)
        own = lab == k
        a[own] = s[own] / max(cnt - 1, 1)
        other = ~own
        b[other] = np.minimum(b[other], s[other] / cnt)
    sing = np.array([(lab == lab[i]).sum() == 1 for i in range(n)])
    sil = np.where(sing, 0.0, (b - a) / np.maximum(np.maximum(a, b), EPS))
    return float(sil.mean())

def calinski(X, lab):
    ks = np.unique(lab); n, k = len(X), len(ks)
    if k < 2 or k >= n: return -1.0
    mu = X.mean(0); B = W = 0.0
    for c in ks:
        Xc = X[lab == c]; mc_ = Xc.mean(0)
        B += len(Xc) * ((mc_ - mu) ** 2).sum(); W += ((Xc - mc_) ** 2).sum()
    return float((B / (k - 1)) / (W / (n - k))) if W > 0 else -1.0

def noise_instance(frac, seed):
    r = np.random.default_rng(seed); c = np.array([[0.0, 0.0], [0.0, 4.0]]); k = int(round(frac * 100))
    X = np.vstack([r.normal(c[0], 1.0, (100, 2)), r.normal(c[1], 1.0, (100, 2)), r.normal(c[0], 3.0, (k, 2))])
    y = np.r_[np.zeros(100), np.ones(100), np.zeros(k)].astype(int)
    return X, y, 2

def imbalance_instance(C, ratio, seed):
    r = np.random.default_rng(seed)
    R = 3.5 / (2 * np.sin(np.pi / C)); ang = 2 * np.pi * np.arange(C) / C
    c = np.c_[R * np.cos(ang), R * np.sin(ang)]
    n = np.maximum(np.round(100 * ratio ** (-np.arange(C) / max(C - 1, 1))).astype(int), 3)
    X = np.vstack([r.normal(c[k], 1.0, (n[k], 2)) for k in range(C)])
    y = np.concatenate([np.full(n[k], k) for k in range(C)])
    return X, y, C

In [ ]:
from PIL import Image
GRID = (1.05, 1.1, 1.3, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0)
PAIRS = [(a, b) for i, a in enumerate(GRID) for b in GRID[i + 1:]]
IT2_KEYS = [(a, b, mode) for (a, b) in PAIRS for mode in ("m1", "mid", "m2")]
DEC_KEYS = [(a, b) for a in GRID for b in GRID]
FAMILIES = {"FCM": list(GRID), "IT2FCM": IT2_KEYS, "IT2FCM-NT": IT2_KEYS, "T1dec": DEC_KEYS}
SYN = ("equal", "volume", "noise", "density", "volume3")
REAL = ("iris", "wine", "wdbc", "wbc", "sonar", "pima", "seeds", "glass", "ionosphere", "banknote", "ecoli")
NOISE = {f"noise{int(f*100):02d}": f for f in (0.0, 0.1, 0.2, 0.4, 0.6)}
IMB = {f"imb_C{C}_r{r}": (C, r) for C in (3, 5, 8) for r in (1, 5, 20)}
N_SYN, N_REAL, N_SWEEP, N_INIT, TOL, N_SIL = 40, 20, 20, 2, 1e-6, 1000

def get_instance(nm, s, data_dir):
    if nm in SYN:
        X, y, c, C = scenario(nm, 1000 + s); return X, y, C
    if nm in NOISE: return noise_instance(NOISE[nm], 6000 + s)
    if nm in IMB: return imbalance_instance(*IMB[nm], 7000 + s)
    X, y, C = real_dataset3(nm, data_dir); r = np.random.default_rng(3000 + s)
    idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], int(round(0.75 * (y == k).sum())), replace=False) for k in range(C)]))
    return X[idx], y[idx], C

def fit(fam, key, X, V0, order):
    if fam == "FCM": return run_fcm(X, V0, key, tol=TOL)
    if fam == "T1dec": return run_fcm_decoupled(X, V0, key[0], key[1], tol=TOL)
    a, b, mode = key; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
    if fam == "IT2FCM":
        V, hp, hn = run_it2fcm_fast(X, V0, a, b, mc=mc, tol=TOL, order=order); return V, hp
    return run_it2fcm_nt(X, V0, a, b, mc=mc, tol=TOL)

def run_instance06(X, y, C, s, evaluate=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    rec = {f: {k: dict(ari=[], acc=[], nmi=[], f=[], sil=[], ch=[]) for k in keys} for f, keys in FAMILIES.items()}
    secs = {f: 0.0 for f in FAMILIES}
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for fam, keys in FAMILIES.items():
            t0 = time.perf_counter()
            for k in keys:
                V, lab = fit(fam, k, X, V0, order)
                sc = evaluate(V, lab) if evaluate else scores(y, lab, C)
                d = rec[fam][k]
                for m in ("ari", "acc", "nmi"): d[m].append(sc[m])
                d["f"].append(sc.get("f", np.nan)); d["sil"].append(silhouette_pre(Dsub, lab[sub])); d["ch"].append(calinski(X, lab))
            secs[fam] += time.perf_counter() - t0
    return dict(rec=rec, sec_per_run={f: secs[f] / (N_INIT * len(FAMILIES[f])) for f in FAMILIES})

def dibco_instance(i, f, g):
    I8 = np.array(Image.open(f).convert("L")); G = np.array(Image.open(g).convert("L")) < 128
    hist_ink = np.bincount(I8[G].ravel(), minlength=256).astype(float)
    hist_bg = np.bincount(I8[~G].ravel(), minlength=256).astype(float)
    r = np.random.default_rng(5000 + i)
    X = (I8.ravel()[r.choice(I8.size, min(2000, I8.size), replace=False)] / 255.0)[:, None]
    def evaluate(V, lab):
        t = float(V.ravel().mean()); k = int(np.clip(np.ceil(t * 255.0), 0, 256))
        tp, fp = hist_ink[:k].sum(), hist_bg[:k].sum(); fn, tn = hist_ink.sum() - tp, hist_bg.sum() - fp
        M = np.array([[tn, fp], [fn, tp]])
        return dict(acc=float((tp + tn) / M.sum()), ari=ari_from_cont(M), nmi=nmi_from_cont(M),
                    f=float(2 * tp / max(2 * tp + fp + fn, 1)))
    out = run_instance06(X, np.zeros(len(X), int), 2, i, evaluate=evaluate)
    out["name"] = os.path.basename(f).replace("TRAIN_", "").replace("_source", "")
    return out

In [ ]:
def hr_bounds_minmax(D, m1, m2):
    U1, U2 = fcm_membership(D, m1), fcm_membership(D, m2)
    return np.minimum(U1, U2), np.maximum(U1, U2)

def run_it2fcm_mm(X, V0, m1, m2, mc=None, iters=300, tol=1e-6, order=None):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    order = np.argsort(X, axis=0, kind="stable") if order is None else order
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds_minmax(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds_minmax(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1)

MM = "IT2FCM-mm"

def run_instance07(X, y, C, s, evaluate=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    rec = {k: dict(ari=[], acc=[], nmi=[], f=[], sil=[], ch=[]) for k in IT2_KEYS}
    t0 = time.perf_counter(); mism_pts = mism_runs = 0
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for k in IT2_KEYS:
            a, b, mode = k; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
            V, hp, hn = run_it2fcm_mm(X, V0, a, b, mc=mc, tol=TOL, order=order)
            sc = evaluate(V, hp) if evaluate else scores(y, hp, C)
            d = rec[k]
            for m in ("ari", "acc", "nmi"): d[m].append(sc[m])
            d["f"].append(sc.get("f", np.nan)); d["sil"].append(silhouette_pre(Dsub, hp[sub])); d["ch"].append(calinski(X, hp))
            mm_ = int((hp != hn).sum()); mism_pts += mm_; mism_runs += int(mm_ > 0)
    return dict(rec={MM: rec}, sec_per_run={MM: (time.perf_counter() - t0) / (N_INIT * len(IT2_KEYS))},
                mism_points=mism_pts, mism_runs=mism_runs)

def dibco_instance07(i, f, g):
    I8 = np.array(Image.open(f).convert("L")); G = np.array(Image.open(g).convert("L")) < 128
    hist_ink = np.bincount(I8[G].ravel(), minlength=256).astype(float)
    hist_bg = np.bincount(I8[~G].ravel(), minlength=256).astype(float)
    r = np.random.default_rng(5000 + i)
    X = (I8.ravel()[r.choice(I8.size, min(2000, I8.size), replace=False)] / 255.0)[:, None]
    def evaluate(V, lab):
        t = float(V.ravel().mean()); k = int(np.clip(np.ceil(t * 255.0), 0, 256))
        tp, fp = hist_ink[:k].sum(), hist_bg[:k].sum(); fn, tn = hist_ink.sum() - tp, hist_bg.sum() - fp
        M = np.array([[tn, fp], [fn, tp]])
        return dict(acc=float((tp + tn) / M.sum()), ari=ari_from_cont(M), nmi=nmi_from_cont(M), f=float(2 * tp / max(2 * tp + fp + fn, 1)))
    out = run_instance07(X, np.zeros(len(X), int), 2, i, evaluate=evaluate)
    out["name"] = os.path.basename(f).replace("TRAIN_", "").replace("_source", "")
    return out

In [ ]:
from scipy import stats
MAIN = SYN + REAL + ("dibco",)
SWEEPS = tuple(NOISE) + tuple(IMB)
FAM5 = dict(FAMILIES); FAM5[MM] = IT2_KEYS
PAIRS5 = (("IT2FCM", "FCM"), ("IT2FCM", "T1dec"),
          (MM, "FCM"), (MM, "T1dec"), (MM, "IT2FCM"), (MM, "IT2FCM-NT"))

def halves(nm, R):
    ids = sorted(R); h = len(ids) // 2
    if nm != "dibco": return ids[:h], ids[h:]
    rng = np.random.default_rng(7); years = {i: R[i]["name"][5:9] for i in ids}; fold = {}
    for yv in sorted(set(years.values())):
        ix = [i for i in ids if years[i] == yv]; ix = [ix[j] for j in rng.permutation(len(ix))]
        for j, i in enumerate(ix): fold[i] = j % 2
    return [i for i in ids if fold[i] == 0], [i for i in ids if fold[i] == 1]

def tp(d):
    d = np.asarray(d, float)
    if len(d) < 2 or np.allclose(d, 0): return dict(mean=float(d.mean()) if len(d) else 0.0, t_p=1.0, w_p=1.0, n=len(d))
    t = float(stats.ttest_1samp(d, 0).pvalue)
    try: w = float(stats.wilcoxon(d).pvalue)
    except ValueError: w = 1.0
    return dict(mean=float(d.mean()), t_p=t if np.isfinite(t) else 1.0, w_p=w if np.isfinite(w) else 1.0, n=len(d))

def verdict(a, b):
    sig = lambda z: z["t_p"] < 0.05 and z["w_p"] < 0.05
    if sig(a) and sig(b) and a["mean"] > 0 and b["mean"] > 0: return "BETTER"
    if sig(a) and sig(b) and a["mean"] < 0 and b["mean"] < 0: return "WORSE"
    return "n.s."

def unsup_value(rec_f, metric, by):
    keys = list(rec_f); nq = len(rec_f[keys[0]][by]); vals = []
    for q in range(nq):
        k = max(keys, key=lambda kk: rec_f[kk][by][q]); vals.append(rec_f[k][metric][q])
    return float(np.mean(vals))

def merge(R06, R07):
    ids = sorted(set(R06) & set(R07)); R = {}
    for i in ids:
        R[i] = dict(R06[i]); R[i]["rec"] = dict(R06[i]["rec"]); R[i]["rec"][MM] = R07[i]["rec"][MM]
        R[i]["sec_per_run"] = dict(R06[i]["sec_per_run"]); R[i]["sec_per_run"][MM] = R07[i]["sec_per_run"][MM]
        R[i]["mism_points"] = R07[i].get("mism_points", 0); R[i]["mism_runs"] = R07[i].get("mism_runs", 0)
    return R

def analyse_dataset(nm, R, metric):
    A, B = halves(nm, R); out = {"n": len(R)}
    for by in ("sil", "ch"):
        val = {f: {i: unsup_value(R[i]["rec"][f], metric, by) for i in R} for f in FAM5}
        cmp = {}
        for (x, z) in PAIRS5:
            a = tp([val[x][i] - val[z][i] for i in A]); b = tp([val[x][i] - val[z][i] for i in B])
            cmp[f"{x} vs {z}"] = dict(A=a, B=b, verdict=verdict(a, b), mean=float(np.mean([val[x][i] - val[z][i] for i in R])))
        out[by] = dict(mean={f: float(np.mean(list(val[f].values()))) for f in FAM5}, cmp=cmp)
    lt = {}
    for sn, (tr, te) in (("A->B", (A, B)), ("B->A", (B, A))):
        sel = {f: max(FAM5[f], key=lambda k: np.mean([np.mean(R[i]["rec"][f][k][metric]) for i in tr])) for f in FAM5}
        test = {f: {i: float(np.mean(R[i]["rec"][f][sel[f]][metric])) for i in te} for f in FAM5}
        lt[sn] = dict(selected={f: str(sel[f]) for f in sel}, cmp={f"{x} vs {z}": tp([test[x][i] - test[z][i] for i in te]) for (x, z) in PAIRS5})
    out["label_tuned"] = dict(splits=lt, verdict={c: verdict(lt["A->B"]["cmp"][c], lt["B->A"]["cmp"][c]) for c in lt["A->B"]["cmp"]},
                              mean={c: float(np.mean([lt[s]["cmp"][c]["mean"] for s in lt])) for c in lt["A->B"]["cmp"]})
    out["sec_per_run"] = {f: float(np.median([R[i]["sec_per_run"][f] for i in R])) for f in FAM5}
    out["mm_partition_vs_nearest"] = dict(points=int(sum(R[i]["mism_points"] for i in R)), runs=int(sum(R[i]["mism_runs"] for i in R)))
    out["mm_fingerprint"] = float(np.mean([np.mean(R[i]["rec"][MM][k]["acc"]) for i in R for k in IT2_KEYS]))
    return out

def dataset_level(RES, names, key):
    rows = [n for n in names if n in RES]; res = {}
    for (x, z) in PAIRS5:
        c = f"{x} vs {z}"
        d = np.array([RES[n]["label_tuned"]["mean"][c] if key == "label_tuned" else RES[n][key]["cmp"][c]["mean"] for n in rows])
        w = float(stats.wilcoxon(d).pvalue) if np.any(np.abs(d) > 1e-12) else 1.0
        res[c] = dict(n=len(rows), median=float(np.median(d)), pos=int((d > 0).sum()), neg=int((d < 0).sum()), wilcoxon_p=w)
    if key != "label_tuned":
        M = np.array([[RES[n][key]["mean"][f] for f in FAM5] for n in rows])
        fr = stats.friedmanchisquare(*M.T); ranks = np.mean([stats.rankdata(-r) for r in M], axis=0)
        res["friedman"] = dict(p=float(fr.pvalue), mean_rank=dict(zip(FAM5, map(float, ranks))))
    return res

In [ ]:
from scipy import stats
ALL = MAIN + SWEEPS
MERGED = {}
for nm in ALL:
    p06, p07 = os.path.join(DIR_06, f"runs06_{nm}.pkl"), os.path.join(DIR_07, f"runs07_{nm}.pkl")
    if os.path.exists(p06) and os.path.exists(p07):
        MERGED[nm] = merge(pickle.load(open(p06, "rb")), pickle.load(open(p07, "rb")))
metric_of = lambda nm, m: "f" if (m == "acc" and nm == "dibco") else m
print("sets:", len(MERGED), "| instances:", sorted({len(R) for R in MERGED.values()}))

def restrict(G):
    G = set(G)
    return {"FCM": [m for m in GRID if m in G],
            "IT2FCM": [k for k in IT2_KEYS if k[0] in G and k[1] in G],
            "IT2FCM-NT": [k for k in IT2_KEYS if k[0] in G and k[1] in G],
            "T1dec": [k for k in DEC_KEYS if k[0] in G and k[1] in G],
            MM: [k for k in IT2_KEYS if k[0] in G and k[1] in G]}
GRIDS = {"full": GRID, "G1": (1.1, 1.3, 1.5, 2.0, 3.0, 5.0, 10.0), "G2": (1.3, 1.5, 2.0, 3.0, 5.0)}

def unsup_g(rec_f, keys, metric, by):
    nq = len(rec_f[keys[0]][by]); vals = []
    for q in range(nq):
        k = max(keys, key=lambda kk: rec_f[kk][by][q]); vals.append(rec_f[k][metric][q])
    return float(np.mean(vals))

def analyse_g(nm, R, metric, fams):
    A, B = halves(nm, R); out = {"n": len(R)}
    val = {f: {i: unsup_g(R[i]["rec"][f], fams[f], metric, "sil") for i in R} for f in fams}
    cmp = {}
    for (x, z) in PAIRS5:
        a = tp([val[x][i] - val[z][i] for i in A]); b = tp([val[x][i] - val[z][i] for i in B])
        cmp[f"{x} vs {z}"] = dict(verdict=verdict(a, b), mean=float(np.mean([val[x][i] - val[z][i] for i in R])))
    out["sil"] = dict(mean={f: float(np.mean(list(val[f].values()))) for f in fams}, cmp=cmp)
    lt = {}
    for sn, (tr, te) in (("A->B", (A, B)), ("B->A", (B, A))):
        sel = {f: max(fams[f], key=lambda k: np.mean([np.mean(R[i]["rec"][f][k][metric]) for i in tr])) for f in fams}
        test = {f: {i: float(np.mean(R[i]["rec"][f][sel[f]][metric])) for i in te} for f in fams}
        lt[sn] = dict(selected={f: str(sel[f]) for f in sel}, cmp={f"{x} vs {z}": tp([test[x][i] - test[z][i] for i in te]) for (x, z) in PAIRS5})
    out["label_tuned"] = dict(splits=lt, verdict={c: verdict(lt["A->B"]["cmp"][c], lt["B->A"]["cmp"][c]) for c in lt["A->B"]["cmp"]},
                              mean={c: float(np.mean([lt[s]["cmp"][c]["mean"] for s in lt])) for c in lt["A->B"]["cmp"]})
    return out

RES = {}
for gname, G in GRIDS.items():
    fams = restrict(G); RES[gname] = {}
    for m in ("acc", "ari"):
        RES[gname][m] = {nm: analyse_g(nm, R, metric_of(nm, m), fams) for nm, R in MERGED.items() if len(R) >= 4}
    print(f"grid {gname}: analyzed", flush=True)

def walsh_hl(d):
    d = np.asarray(d, float); i, j = np.triu_indices(len(d))
    return float(np.median((d[i] + d[j]) / 2))
def boot_ci(d, B=20000, seed=35):
    d = np.asarray(d, float); r = np.random.default_rng(seed)
    hl = [walsh_hl(d[r.integers(0, len(d), len(d))]) for _ in range(B)]
    return float(np.percentile(hl, 2.5)), float(np.percentile(hl, 97.5))
def tost(d, delta):
    d = np.asarray(d, float)
    try:
        pl = float(stats.wilcoxon(d + delta, alternative="greater").pvalue)
        pu = float(stats.wilcoxon(d - delta, alternative="less").pvalue)
    except ValueError:
        return 1.0
    return max(pl, pu)
def perset(gname, key, metric, c):
    names = [n for n in MAIN if n in RES[gname][metric]]
    if key == "label_tuned": return np.array([RES[gname][metric][n]["label_tuned"]["mean"][c] for n in names]) * 100
    return np.array([RES[gname][metric][n]["sil"]["cmp"][c]["mean"] for n in names]) * 100
EQUIV = {}
for key, metric in (("label_tuned", "acc"), ("label_tuned", "ari"), ("sil", "ari")):
    for c in (f"{MM} vs FCM", f"{MM} vs T1dec", "IT2FCM vs T1dec", f"{MM} vs IT2FCM", f"{MM} vs IT2FCM-NT"):
        d = perset("full", key, metric, c); lo, hi = boot_ci(d)
        EQUIV[f"{key}:{metric}:{c}"] = dict(n=len(d), hl=walsh_hl(d), ci95=(lo, hi), wilcoxon_p=float(stats.wilcoxon(d).pvalue) if np.any(d != 0) else 1.0,
                                            tost={str(dl): tost(d, dl) for dl in (0.5, 1.0, 2.0)})

HR_M = [m for m in GRID if 1.1 <= m <= 10.0]
def per_fuzzifier(nm, fam, mt):
    R = MERGED[nm]; ids = list(R); out = {}
    for m in HR_M:
        fcm = np.mean([np.mean(R[i]["rec"]["FCM"][m][mt]) for i in ids])
        if fam == "T1dec": keys = [k for k in DEC_KEYS if (k[0] == m or k[1] == m) and 1.1 <= k[0] <= 10 and 1.1 <= k[1] <= 10]
        else: keys = [k for k in IT2_KEYS if k[2] == "mid" and (k[0] == m or k[1] == m) and 1.1 <= k[0] <= 10 and 1.1 <= k[1] <= 10]
        best = max(np.mean([np.mean(R[i]["rec"][fam][k][mt]) for i in ids]) for k in keys)
        out[m] = float(best - fcm)
    return out
HR = {}
for nm in MAIN:
    if nm not in MERGED: continue
    mt = metric_of(nm, "acc"); HR[nm] = {fam: per_fuzzifier(nm, fam, mt) for fam in (MM, "IT2FCM", "T1dec")}
    R = MERGED[nm]
    for fam in (MM, "IT2FCM"):
        for mref in (2.0, 10.0):
            d = [np.mean(R[i]["rec"][fam][(2.0, 10.0, "mid")][mt]) - np.mean(R[i]["rec"]["FCM"][mref][mt]) for i in R]
            HR[nm][f"{fam} (2,10) vs FCM m={mref:g}"] = dict(mean=float(np.mean(d)), t_p=float(stats.ttest_1samp(d, 0).pvalue) if np.std(d) > 0 else 1.0)

def edge_share():
    out = {}
    for f in FAM5:
        n = e = 0
        for nm, R in MERGED.items():
            for i in R:
                rec = R[i]["rec"][f]; keys = FAM5[f]; nq = len(rec[keys[0]]["sil"])
                for q in range(nq):
                    k = max(keys, key=lambda kk: rec[kk]["sil"][q]); kk = k if isinstance(k, tuple) else (k,)
                    n += 1; e += any(v in (1.05, 20.0) for v in kk[:2])
        lt = [RES["full"]["acc"][nm]["label_tuned"]["splits"][s]["selected"][f] for nm in RES["full"]["acc"] for s in ("A->B", "B->A")]
        out[f] = dict(label_free_edge=e / n, label_tuned_edge=float(np.mean([("1.05" in s) or ("20.0" in s) for s in lt])))
    return out
EDGE = edge_share()
DLG = {g: {(k, m): dataset_level(RES[g][m], MAIN, k) for k in ("label_tuned",) for m in ("acc", "ari")} for g in GRIDS}
for g in GRIDS:
    DLG[g][("sil", "ari")] = dataset_level(RES[g]["ari"], MAIN, "sil")

In [ ]:
LINES = []
def say(s=""): print(s); LINES.append(s)
say("=" * 96); say(f"09 -- equivalence, grid sensitivity and the per-fuzzifier design   (run {RUN_STAMP})"); say("=" * 96)
say("1. EQUIVALENCE across the 17 main sets (points; HL = Hodges-Lehmann median difference with 95% bootstrap CI; TOST = p of two one-sided Wilcoxon tests at each margin)")
for k, e in EQUIV.items():
    t = ", ".join(f"+-{dl}: {p:.3f}" for dl, p in e["tost"].items())
    say(f"  {k:48s} HL {e['hl']:+6.2f}  CI [{e['ci95'][0]:+6.2f}, {e['ci95'][1]:+6.2f}]  Wilcoxon p {e['wilcoxon_p']:.3f} | TOST {t}")
say("")
say("2. PER-FUZZIFIER DESIGN OF HWANG AND RHEE (m from 1.1 to 10)")
for fam in (MM, "IT2FCM", "T1dec"):
    cells = [HR[nm][fam][m] for nm in HR for m in HR_M]
    allm = sum(all(HR[nm][fam][m] >= 0 for m in HR_M) for nm in HR)
    say(f"  {fam:10s}: >= FCM in {np.mean(np.array(cells) >= 0)*100:5.1f}% of (set, m) cells; at every m on {allm}/{len(HR)} sets; mean gain {np.mean(cells)*100:+.2f} pts")
for fam in (MM, "IT2FCM"):
    for mref in (2.0, 10.0):
        k = f"{fam} (2,10) vs FCM m={mref:g}"; v = [HR[nm][k] for nm in HR]
        say(f"    {k:34s}: better (t-test p<0.05) on {sum(x['mean'] > 0 and x['t_p'] < 0.05 for x in v)}, worse on {sum(x['mean'] < 0 and x['t_p'] < 0.05 for x in v)} of {len(v)}")
say("")
say("3. RESTRICTED GRIDS (full; G1 = 1.1 to 10; G2 = 1.3 to 5)")
for g in GRIDS:
    for (key, m), lab in (((("label_tuned", "acc")), "label-tuned acc"), ((("label_tuned", "ari")), "label-tuned ARI"), ((("sil", "ari")), "label-free ARI")):
        parts = []
        for c in (f"{MM} vs FCM", f"{MM} vs T1dec", "IT2FCM vs T1dec"):
            v = [(RES[g][m][n]["label_tuned"]["verdict"][c] if key == "label_tuned" else RES[g][m][n]["sil"]["cmp"][c]["verdict"]) for n in MAIN if n in RES[g][m]]
            parts.append(f"{c.replace('IT2FCM-mm', 'IT2').replace('IT2FCM', 'IT2p').replace('T1dec', 'T1d')}: +{v.count('BETTER')}/-{v.count('WORSE')} p={DLG[g][(key, m)][c]['wilcoxon_p']:.3f}")
        extra = f" | Friedman p={DLG[g][(key, m)]['friedman']['p']:.3f}" if key == "sil" else ""
        say(f"  {g:4s} {lab:16s} " + " ; ".join(parts) + extra)
say("    settings selected at the edge of the grid (1.05 or 20):")
for f, e in EDGE.items():
    say(f"    {f:10s}: label-tuned {e['label_tuned_edge']*100:5.1f}%   label-free (silhouette) {e['label_free_edge']*100:5.1f}%")
say("")
say("4. IT2 AGAINST IT2-NT (label-tuned; sets with a verdict or a difference of at least 2 points)")
for nm in list(MAIN) + [n for n in SWEEPS if n.startswith("imb")]:
    m = "acc" if nm in MAIN else "ari"
    if nm not in RES["full"][m]: continue
    r = RES["full"][m][nm]["label_tuned"]; c = f"{MM} vs IT2FCM-NT"
    if r["verdict"][c] != "n.s." or abs(r["mean"][c]) >= 0.02:
        say(f"  {nm:12s} IT2 - IT2-NT {r['mean'][c]*100:+6.2f}  {r['verdict'][c]}")
payload = dict(notebook="09", run_stamp=RUN_STAMP, equivalence=EQUIV,
               hr_design={nm: {str(k): v for k, v in d.items()} for nm, d in HR.items()}, grid_edge=EDGE,
               grid_results=RES, grid_dataset_level={g: {f"{k[0]}:{k[1]}": v for k, v in d.items()} for g, d in DLG.items()})
report = "\n".join(LINES) + "\n"
paths = [os.path.join(OUTDIR, "results_09.json"), os.path.join(OUTDIR, "results_09_report.txt"),
         os.path.join(OUTDIR, "reports", f"results_09_{RUN_STAMP}.json"), os.path.join(OUTDIR, "reports", f"results_09_report_{RUN_STAMP}.txt")]
for p in paths:
    with open(p, "w") as fh:
        if p.endswith(".json"): json.dump(payload, fh, indent=1, default=str)
        else: fh.write(report)
        fh.flush(); os.fsync(fh.fileno())
print()
for p in paths:
    print(f"{'written' if os.path.exists(p) and os.path.getsize(p) > 0 else 'NOT WRITTEN'}: {p}")